# Week 6: Data Retrieval and Processing
This notebook implements the data retrieval and preprocessing pipeline for Milestone 1 IoT data stored on the blockchain.

In [1]:
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from web3 import Web3

def get_env_value(key, default=None, env_path=".env"):
    # Prefer shell environment variables, fallback to local .env
    value = os.getenv(key)
    if value:
        return value

    if os.path.exists(env_path):
        with open(env_path, "r", encoding="utf-8") as env_file:
            for line in env_file:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                name, raw_value = line.split("=", 1)
                if name.strip() == key:
                    return raw_value.strip().strip('"').strip("'")
    return default

def get_env_int(key, default):
    return int(get_env_value(key, str(default)))

def get_env_float(key, default):
    return float(get_env_value(key, str(default)))

# Connect to local Ganache blockchain
ganache_url = get_env_value("GANACHE_URL", "http://127.0.0.1:8545")
web3 = Web3(Web3.HTTPProvider(ganache_url))

if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
else:
    print("❌ Connection failed. Ensure Ganache is running.")

✅ Connected to Ganache successfully!


In [2]:
# Load deployed smart contract configuration
contract_address = get_env_value("CONTRACT_ADDRESS")
if not contract_address:
    raise ValueError("CONTRACT_ADDRESS is missing. Set it in .env or the environment.")
contract_address = Web3.to_checksum_address(contract_address)

abi_path = Path(get_env_value("ABI_PATH", "contracts/abi.json"))

# Load ABI
with open(abi_path, "r", encoding="utf-8") as abi_file:
    abi = json.load(abi_file)

# Instantiate the contract
contract = web3.eth.contract(address=contract_address, abi=abi)

# Configure default account
contract_owner = contract.functions.owner().call()
if contract_owner not in web3.eth.accounts:
    override_owner = get_env_value("CONTRACT_OWNER")
    if override_owner:
        contract_owner = Web3.to_checksum_address(override_owner)
    else:
        contract_owner = web3.eth.accounts[0]

web3.eth.default_account = contract_owner

print(f"✅ Connected to Smart Contract at {contract_address}")
print(f"✅ Using default sender account: {web3.eth.default_account}")

✅ Connected to Smart Contract at 0x3c64Bb4df9DC16b57D62B281bd56742060CF78Ee
✅ Using default sender account: 0x384F585463b9D2288A637615e1576E4F3798B073


In [3]:
# Get the total number of stored records from blockchain
total_records = contract.functions.getTotalRecords().call()
print(f"Total IoT records stored: {total_records}")

# Retrieve and print the first stored record to verify retrieval works
if total_records > 0:
    first_record = contract.functions.getRecord(0).call()
    first_package = first_record[1]

    # Load static details from raw CSV if available to enrich the first record print
    csv_path = get_env_value("CSV_PATH")
    if csv_path and os.path.exists(csv_path):
        csv_df = pd.read_csv(csv_path)
        row = csv_df[csv_df['package_id'] == first_package].iloc[0]
        
        # Query blockchain for all telemetry fields of this package
        pkg_telemetry = {}
        for i in range(total_records):
            rec = contract.functions.getRecord(i).call()
            if rec[1] == first_package:
                pkg_telemetry[rec[2]] = rec[3]
                
        # Extract fields
        order_date = row.get('order_date', 'N/A')
        delivery_date = row.get('expected_delivery_date', row.get('delivery_date', 'N/A'))
        origin = row.get('origin', 'N/A')
        current_loc = pkg_telemetry.get('Location', row.get('current_location', 'N/A'))
        delivery_loc = row.get('delivery_location', 'N/A')
        perishable = row.get('perishable', 'N/A')
        temp_str = pkg_telemetry.get('Temperature', f"{row.get('temperature', 0.0)}°C")

        # Calculate temperature issue
        temp_val = float(row.get('temperature', 0.0))
        if str(perishable).strip().lower() == 'yes':
            temp_issue = "Normal" if temp_val <= 8.0 else "Temperature Alert"
        else:
            temp_issue = "Not Applicable"

        status = pkg_telemetry.get('Status', row.get('latest_status', 'N/A'))

        pivoted_first_record = [
            first_record[0],
            first_package,
            order_date,
            delivery_date,
            origin,
            current_loc,
            delivery_loc,
            perishable,
            temp_str,
            temp_issue,
            status
        ]

        print("First Stored Record:", pivoted_first_record)
        print("\n📦 First Stored Record")
        print(f"Timestamp: {pivoted_first_record[0]}")
        print(f"Package ID: {pivoted_first_record[1]}")
        print(f"Location: {pivoted_first_record[5]}")
        print(f"Status: {pivoted_first_record[10]}")
    else:
        print("First Stored Record:", first_record)
else:
    print("No records stored yet on the blockchain.")

Total IoT records stored: 400


First Stored Record: [1780658433, 'PKG7545', '2026-04-29 23:26:26.858009', '2026-05-05 23:26:26.858015', 'Tokyo', 'Naha Central Post Office', 'Tokyo', 'No', '10.7°C', 'Not Applicable', 'Out for Delivery']

📦 First Stored Record
Timestamp: 1780658433
Package ID: PKG7545
Location: Naha Central Post Office
Status: Out for Delivery


In [4]:
# Fetch all stored IoT data and structure it in a DataFrame
data = []
for i in range(total_records):
    record = contract.functions.getRecord(i).call()
    data.append({
        "timestamp": record[0],
        "device_id": record[1],
        "data_type": record[2],
        "data_value": record[3]
    })

# Convert retrieved blockchain records to a DataFrame
df_events = pd.DataFrame(data)

# Convert timestamp values to readable datetime format
df_events["timestamp"] = pd.to_datetime(df_events["timestamp"], unit="s")

# Extract numerical values from data_value for sensor readings such as temperature and humidity
df_events["numeric_value"] = df_events["data_value"].str.extract(r'(-?\d+\.?\d*)').astype(float)

# Pivot retrieved blockchain events by package/device ID to structure them as wide logistics records
df = df_events.pivot_table(
    index=["device_id"],
    columns="data_type",
    values=["data_value", "numeric_value"],
    aggfunc="first"
)

# Flatten MultiIndex columns into single-level column names
df.columns = [f"{col[1]}_{col[0]}".lower() for col in df.columns]

# Map the earliest blockchain timestamp for each package ID
df["timestamp"] = df.index.map(
    df_events.groupby("device_id")["timestamp"].min()
)

# Reset index to make device_id a regular column
df = df.reset_index()

# Rename fields for logistics readability
rename_cols = {
    "status_data_value": "status",
    "temperature_numeric_value": "temperature_celsius",
    "temperature_data_value": "temperature",
    "humidity_numeric_value": "humidity",
    "location_data_value": "current_location"
}
df = df.rename(columns={k: v for k, v in rename_cols.items() if k in df.columns})
df = df.rename(columns={"device_id": "package_id"})

# Load static details from the raw CSV to enrich blockchain-based records
csv_path = get_env_value("CSV_PATH")
if csv_path and os.path.exists(csv_path):
    csv_df = pd.read_csv(csv_path)

    # Merge available shipment context fields from the raw dataset
    enrich_cols = ["package_id"]
    for col in ["order_date", "expected_delivery_date", "delivery_date", "origin", "delivery_location", "perishable"]:
        if col in csv_df.columns:
            enrich_cols.append(col)

    csv_subset = csv_df[enrich_cols]
    df = pd.merge(df, csv_subset, on="package_id", how="left")

# Calculate temperature issue labels for perishable packages
if "temperature_celsius" in df.columns and "perishable" in df.columns:
    def check_temp_issue(row):
        perish = str(row.get("perishable", "")).strip().lower()
        if perish == "yes":
            try:
                temp = float(row.get("temperature_celsius", 0.0))
                return "Normal" if temp <= 8.0 else "Temperature Alert"
            except Exception:
                return "Normal"
        return "Not Applicable"

    df["temperature_issue"] = df.apply(check_temp_issue, axis=1)

# Standardize detailed courier status values into five package status categories
def standardize_package_status(status):
    status_text = str(status).strip().lower()

    if any(keyword in status_text for keyword in ["delivered", "hand it over"]):
        return "Delivered"

    if "returned" in status_text:
        return "Cancelled"

    if any(keyword in status_text for keyword in [
        "delay", "under investigation", "absence", "unknown", "failure"
    ]):
        return "Delayed"

    if any(keyword in status_text for keyword in ["storage", "hold"]):
        return "To Ship"

    return "In Transit"

if "status" in df.columns:
    df["status"] = df["status"].apply(standardize_package_status)

# Remove redundant temperature_celsius column because the temperature column is retained
df = df.drop(columns=["temperature_celsius"], errors="ignore")

# Reorder columns logically for the final cleaned CSV output
cols_order = [
    "timestamp", "package_id", "order_date", "expected_delivery_date", "delivery_date",
    "origin", "current_location", "delivery_location", "temperature", "humidity",
    "perishable", "temperature_issue", "status"
]
df = df[[col for col in cols_order if col in df.columns]]

# Display first few records
print("Retrieved Blockchain DataFrame Preview:")
display(df.head())

print("\nFinal status categories:")
print(df["status"].value_counts() if "status" in df.columns else "Status column not found")

print("\nFinal columns:")
print(df.columns.tolist())

Retrieved Blockchain DataFrame Preview:


,timestamp,package_id,order_date,expected_delivery_date,origin,current_location,delivery_location,temperature,temperature_celsius,humidity,perishable,temperature_issue,status
0,2026-06-18 12:34:36,PKG1191,2026-05-02 23:26:26.862041,2026-05-07 23:26:26.862045,Osaka,Nagoya Central Post Office,Kyoto,-2.0°C,-2.0,74.0,No,Not Applicable,Storage
1,2026-06-18 12:34:54,PKG1198,2026-05-02 23:26:26.867367,2026-05-05 23:26:26.867371,Osaka,Osaka Central Post Office,Yokohama,-3.4°C,-3.4,30.0,No,Not Applicable,Hold at Yamato
2,2026-06-18 12:34:45,PKG1329,2026-05-02 23:26:26.864535,2026-05-06 23:26:26.864538,Osaka,Osaka Central Post Office,Kyoto,7.9°C,7.9,80.0,Yes,Normal,In Transit
3,2026-06-05 11:20:36,PKG1347,2026-04-30 23:26:26.859972,2026-05-07 23:26:26.859975,Nagoya,Nagoya Central Post Office,Tokyo,18.8°C,18.8,76.0,No,Not Applicable,Returned to the sender
4,2026-06-18 12:34:31,PKG1643,2026-04-30 23:26:26.860885,2026-05-05 23:26:26.860888,Fukuoka,Sapporo Central Post Office,Tokyo,14.8°C,14.8,74.0,No,Not Applicable,Arrival


In [5]:
# Data Preprocessing and Cleaning
print("Identifying missing values before cleaning:")
print(df.isna().sum())

# Handle missing values (if any)
# To avoid pandas typing errors, fill numeric columns with 0, and string/object columns with "0"
fill_values = {}
for col in df.columns:
    if df[col].dtype == object or isinstance(df[col].dtype, pd.StringDtype):
        fill_values[col] = "0"
    else:
        fill_values[col] = 0

df.fillna(value=fill_values, inplace=True)

# Display cleaned data
print("\nCleaned and preprocessed data preview:")
display(df.head())

Identifying missing values before cleaning:
timestamp                 0
package_id                0
order_date                0
expected_delivery_date    0
origin                    0
current_location          0
delivery_location         0
temperature               0
temperature_celsius       0
humidity                  0
perishable                0
temperature_issue         0
status                    0
dtype: int64

Cleaned and preprocessed data preview:


,timestamp,package_id,order_date,expected_delivery_date,origin,current_location,delivery_location,temperature,temperature_celsius,humidity,perishable,temperature_issue,status
0,2026-06-18 12:34:36,PKG1191,2026-05-02 23:26:26.862041,2026-05-07 23:26:26.862045,Osaka,Nagoya Central Post Office,Kyoto,-2.0°C,-2.0,74.0,No,Not Applicable,Storage
1,2026-06-18 12:34:54,PKG1198,2026-05-02 23:26:26.867367,2026-05-05 23:26:26.867371,Osaka,Osaka Central Post Office,Yokohama,-3.4°C,-3.4,30.0,No,Not Applicable,Hold at Yamato
2,2026-06-18 12:34:45,PKG1329,2026-05-02 23:26:26.864535,2026-05-06 23:26:26.864538,Osaka,Osaka Central Post Office,Kyoto,7.9°C,7.9,80.0,Yes,Normal,In Transit
3,2026-06-05 11:20:36,PKG1347,2026-04-30 23:26:26.859972,2026-05-07 23:26:26.859975,Nagoya,Nagoya Central Post Office,Tokyo,18.8°C,18.8,76.0,No,Not Applicable,Returned to the sender
4,2026-06-18 12:34:31,PKG1643,2026-04-30 23:26:26.860885,2026-05-05 23:26:26.860888,Fukuoka,Sapporo Central Post Office,Tokyo,14.8°C,14.8,74.0,No,Not Applicable,Arrival


In [6]:
# Ensure that the assets directory exists before exporting files
Path("assets").mkdir(parents=True, exist_ok=True)

# Save cleaned IoT data to a CSV file in the assets/ directory
output_path = "assets/cleaned_iot_data.csv"
df.to_csv(output_path, index=False)
print(f"✅ Cleaned IoT data saved successfully as {output_path}")

# Create another copy of it named "MO-IT148 Homework: Data Retrieval and Processing S3101 Team Kaizen.csv" in assets folder
homework_output_path = "assets/MO-IT148 Homework: Data Retrieval and Processing S3101 Team Kaizen.csv"
df.to_csv(homework_output_path, index=False)
print(f"✅ Homework copy saved successfully as {homework_output_path}")

✅ Cleaned IoT data saved successfully as assets/cleaned_iot_data.csv
✅ Homework copy saved successfully as assets/MO-IT148 Homework: Data Retrieval and Processing S3101 Team Kaizen.csv
